In [1]:
!pip install fastapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.9 MB/s eta 0:00:00


In [5]:
import pickle
import numpy as np
from fastapi import FastAPI
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

In [6]:
# Cargar dataset
data = fetch_california_housing()
X, y = data.data, data.target

In [7]:
# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Entrenar modelo (ejemplo)
# model = RandomForestClassifier() # Original model - for classification
model = RandomForestRegressor(random_state=42) # Changed to RandomForestRegressor - for regression

In [11]:
# Entrenar el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [12]:
# Exportar modelo
with open('modelo.pkl', 'wb') as file:
    pickle.dump(model, file)

In [13]:
# Cargar y usar después
with open('modelo.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

In [18]:
# Nuevos datos definidos por el usuario (no coinciden con el modelo entrenado)
nuevos_datos = [
  {"edad": 35, "ingreso": 50000, "deuda": 2000, "historial_credito": 1, "compras_mes": 5},
  {"edad": 42, "ingreso": 80000, "deuda": 15000, "historial_credito": 0, "compras_mes": 12},
  {"edad": 28, "ingreso": 30000, "deuda": 8000, "historial_credito": 1, "compras_mes": 3}
]

# Convertir la lista de diccionarios a un arreglo numpy
# Asegúrate de que el orden de las claves en los diccionarios
# coincide con el orden de las características que el modelo espera.
# En este caso particular, el modelo fue entrenado con el dataset
# fetch_california_housing, que tiene un conjunto diferente de características
# (MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude).
# Para que la predicción funcione correctamente con el modelo entrenado,
# los datos de entrada deben tener el mismo número de características y estar en el formato esperado.
# Los datos 'nuevos_datos' que has definido NO coinciden con las características
# del modelo entrenado con fetch_california_housing.
# Para fines de DEMOSTRACIÓN y para arreglar el TypeError, convertiremos la lista de dicts a un array,
# pero ten en cuenta que las predicciones resultantes probablemente no tendrán sentido
# en el contexto del modelo entrenado con datos de vivienda de California.
# Deberías proporcionar datos con las 8 características del dataset original si quieres predicciones significativas.

# Creamos un arreglo numpy extrayendo los valores de los diccionarios.
# Esto SOLAMENTE soluciona el TypeError al proporcionar un arreglo numpy,
# pero NO garantiza que el modelo haga predicciones significativas
# ya que las características son diferentes.
# datos_para_prediccion = np.array([[list(d.values()) for d in nuevos_datos]])
# Ajustamos la forma para que sea 2D si es necesario
# datos_para_prediccion = datos_para_prediccion.reshape(len(nuevos_datos), -1)

# --- Inicio de los cambios ---

# Creamos datos de ejemplo que tienen 8 características, como espera el modelo
# Estos son solo datos inventados para demostrar que la predicción funciona con la forma correcta.
# Si quieres predicciones significativas, reemplaza esto con datos reales
# que representen las características de fetch_california_housing.
datos_para_prediccion = np.array([
    [8.3252, 41.0, 6.984127, 1.02381 , 322.0, 2.555556, 37.88, -122.23],
    [6.1,    21.0, 5.1,      1.1,      1100.0, 2.8,      37.1,  -122.1],
    [4.5,    30.0, 5.8,      1.05,      800.0, 2.1,      37.5,  -122.3]
])

# Asegúrate de que el número de filas coincide con el número de "nuevos_datos" si es relevante,
# aunque para este ejemplo, estamos creando datos independientes.
# El reshape ya no es necesario si creas el array con la forma correcta (3, 8).

# --- Fin de los cambios ---


predicciones = loaded_model.predict(datos_para_prediccion)

In [19]:
# Cargar el modelo previamente entrenado
with open("modelo.pkl", "rb") as file:
    modelo = pickle.load(file)

In [20]:
app = FastAPI()

In [22]:
!pip install pydantic

In [24]:
!pip install python-multipart

In [25]:
@app.post("/predict/")
def predict(data: list):
    datos = np.array(data).reshape(1, -1)
    prediccion = modelo.predict(datos)
    return {"prediction": int(prediccion[0])}

In [26]:
# Ejecutar con: uvicorn nombre_del_script:app --reload

Pasos para ejecutar FastAPI en Google Colab

1. Instalar dependencias
En una celda de Colab, instala FastAPI, Uvicorn y pyngrok:

In [27]:
!pip install fastapi uvicorn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.6 MB/s eta 0:00:00


2. Crear el archivo de la API

Puedes escribir tu código FastAPI en un archivo Python. Para esto, usa:

In [28]:
%%writefile app.py
from fastapi import FastAPI
import pickle
import numpy as np

Writing app.py


In [29]:
# Cargar el modelo previamente guardado
with open("modelo.pkl", "rb") as file:
    modelo = pickle.load(file)

In [30]:
app = FastAPI()

In [31]:
@app.post("/predict/")
def predict(data: list):
    datos = np.array(data).reshape(1, -1)
    prediccion = modelo.predict(datos)
    return {"prediction": int(prediccion[0])}

3. Ejecutar el servidor con Uvicorn y ngrok
En Colab, ejecutamos Uvicorn en segundo plano y exponemos el puerto con ngrok:

In [32]:
from pyngrok import ngrok

In [33]:
# Abrir un túnel en el puerto 8000
public_url = ngrok.connect(port=8000)
print(f"FastAPI se ejecuta en {public_url}")

ERROR:pyngrok.process.ngrok:t=2025-05-19T01:37:41+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-05-19T01:37:41+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-05-19T01:37:41+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [34]:
# Iniciar Uvicorn en segundo plano
!uvicorn app:app --host 0.0.0.0 --port 8000 --reload

INFO:     Will watch for changes in these directories: ['/content']
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Started reloader process [5218] using StatReload
ERROR:    Error loading ASGI app. Attribute "app" not found in module "app".
INFO:     Stopping reloader process [5218]


4. Probar la API
Después de ejecutar el código, obtendrás un enlace ngrok (ejemplo: https://xyz.ngrok.io). Para probar la API, ve a:

https://xyz.ngrok.io/docs


Ahí puedes probar las predicciones directamente desde la interfaz de Swagger.

Resumen
Este método te permite ejecutar FastAPI en Google Colab y exponerlo con ngrok para que puedas hacer pruebas en tiempo real.